<a id="top"></a>
# Searching MAST Catalogs with Astroquery
***
## Learning Goals

By the end of this tutorial, you will:

- Discover collections and catalogs available through the MAST TAP Service.
- Understand how to inspect catalog metadata, including column information and query capabilities.
- Learn how to construct queries using column-based criteria and spatial parameters.

## Table of Contents
* [Introduction](#introduction)

* [Imports](#imports)

* [Collections and Catalogs](#collections-and-catalogs)

* [Discovering Collections and Catalogs](#discovering-collections-and-catalogs)

  * [Discovering Available Collections](#discovering-available-collections)

  * [Discovering Catalogs within a Collection](#discovering-catalogs-within-a-collection)

* [Inspecting Catalog Metadata](#inspecting-catalog-metadata)

   * [Inspecting Catalog Columns](#inspecting-catalog-columns)
   
   * [Inspecting Catalog Capabilities](#inspecting-catalog-capabilities)

* [Querying Catalogs](#querying-catalogs)

   * [Shared Query Parameters](#shared-query-parameters)

   * [Criteria Syntax](#criteria-syntax)

   * [Spatial Query Parameters](#positional-query-parameters)

     * [Specifying Spatial Regions](#specifying-spatial-regions)

        * [Cone Search](#cone-search)

        * [Polygon Search](#polygon-search)

   * [Counting Results](#counting-results)

* [Exercises](#exercises)

* [Exercise Solutions](#exercise-solutions)

* [Additional Resources](#additional-resources)

## Introduction

Welcome! This tutorial explores the capabilities of the `astroquery.mast.Catalogs` class, a versatile tool for discovering and querying the wide range of astronomical catalogs hosted by the [Mikulski Archive for Space Telescopes (MAST)](https://archive.stsci.edu/). `Catalogs` is a Python wrapper for our [MAST Table Access Protocol (TAP) Service](https://mast.stsci.edu/vo-tap/), which allows you to query for catalog metadata and data. If you were querying the MAST TAP service directly, you would need to write your queries in [Astronomical Data Query Language (ADQL)](https://www.ivoa.net/documents/latest/ADQL.html). With `astroquery.mast.Catalogs`, you can construct and execute these queries using a more intuitive Python interface, without needing to learn ADQL syntax.

The catalogs available through MAST are diverse, covering a wide range of astronomical objects and phenomena. They include data from various missions and surveys, such as the Hubble Space Telescope, Kepler, TESS, Gaia, and many more. These catalogs are organized into **collections**, each of which may contain one or more catalogs with distinct schemas and capabilities. The `Catalogs` interface is designed for flexible querying of catalog data, including both positional and non-positional queries, as well as the ability to filter results based on specific criteria.

At a high level, querying MAST catalogs with `astroquery.mast.Catalogs` involves the following steps:
1. **Discover** available collections and catalogs.
2. **Inspect** catalog metadata to understand available columns and data types, as well as the capabilities of each catalog.
3. **Query** the catalog using positional and/or non-positional criteria to retrieve relevant data.

In this tutorial, we will cover each of these steps in detail, providing examples and exercises to help you become proficient in using `astroquery.mast.Catalogs` for your astronomical research. Let's get started!

## Imports
This notebook uses the following packages:

- *astroquery.mast* to query MAST catalogs.
- *astropy.units* to specify units for spatial queries.
- *astropy.coordinates* to specify coordinates for spatial queries.
- *regions* to define spatial regions for queries.

In [ ]:
import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.mast import Catalogs
from regions import CircleSkyRegion, PolygonSkyRegion

***

## Collections and Catalogs

MAST catalogs are organized into **collections**, where each collection represents a set of related catalogs with a shared scientific or mission context (e.g., Hubble source catalogs, Gaia data releases, etc.). Within a collection, one or more catalogs may be available, each with its own set of columns and capabilities.

The `Catalogs` class stores a `collection` and `catalog` as attributes. If no collection and/or catalog is specified in a query, these attributes will be used as defaults. 

The default value for the `collection` attribute is "hsc", referring to the [Hubble Source Catalog version 3](https://archive.stsci.edu/hst/hsc/). The default value for `catalog` is "dbo.SumMagAper2CatView". This is a summary source catalog with data describing sources detected in Hubble images, including their positions, magnitudes, and other properties. We'll be using this catalog for other examples in this tutorial.

In [ ]:
# Print the default collection and catalog
print("Default collection:", Catalogs.collection)
print("Default catalog:", Catalogs.catalog)

These attributes can be set with parameters when instantiating a `Catalogs` object, or they can be changed at any time after instantiation to set new defaults for subsequent queries. Both `collection` and `catalog` will be validated when set. `collection` must be a valid collection name, and `catalog` must be a valid catalog within the specified collection. When changing the collection, the catalog will be reset to the default catalog for the new collection.

Here, we'll change the value of `collection` to "ullyses", referring to the [Hubble Ultraviolet Legacy Library of Young Stars as Essential Standards (ULLYSES)](https://ullyses.stsci.edu/) program. The default catalog for this collection is "sciencemetadata", which contains metadata about the scientific exposures taken as part of the ULLYSES program, including their coordinates, observation dates, instruments used, and other properties.

In [ ]:
# Set the collection to a new value
Catalogs.collection = "ullyses"
print("Updated collection:", Catalogs.collection)
print("Updated catalog:", Catalogs.catalog)

You can also create multiple instances of `Catalogs` with different defaults, which can be useful for working with multiple catalogs in the same script or notebook.

In [ ]:
# Catalogs object for the 'hsc' collection
hsc_catalog = Catalogs(collection="hsc")
print("HSC collection:", hsc_catalog.collection)
print("HSC catalog:", hsc_catalog.catalog)

# Catalogs object for the 'ullyses' collection and 'sciencemetadata' catalog
ullyses_catalog = Catalogs(collection="ullyses", catalog="sciencemetadata")
print("\nUllyses collection:", ullyses_catalog.collection)
print("Ullyses catalog:", ullyses_catalog.catalog)

## Discovering Collections and Catalogs

### Discovering Available Collections

To list all of the catalog collections that are accessible via the MAST TAP Service, use the `get_collections()` method. This returns an `astropy.table.Table` containing the names of all available collections.

In [ ]:
# Get a table of available collections
Catalogs.get_collections()

Wow, we have so many catalog collections at our disposal! Think of all the querying that might be possible!

### Discovering Catalogs within a Collection

Once a collection is selected, you can discover which catalogs are available within that collection using the `get_catalogs()` method. This returns an `astropy.table.Table` containing names and descriptions of the catalogs within the currently selected collection.

We'll use this as an opportunity to show our default collection in action. We'll set the collection attribute to "ullyses", so when we call `get_catalogs()`, it will return the catalogs available within that collection.

In [ ]:
# Make sure the collection is set to 'ullyses' before getting catalogs
Catalogs.collection = "ullyses"

# Get catalogs for the 'ullyses' collection
Catalogs.get_catalogs()

To query catalogs for a different collection without changing the class state, you can also specify the collection as a parameter to `get_catalogs()`. This applies to all `Catalogs` methods that take `collection` and/or `catalog` parameters - if specified, those parameters will override the default values stored in the class attributes for that query only, without changing the class state. For most of this tutorial, we'll be passing in the ``collection`` and ``catalog`` parameters explicitly so as to avoid confusion.

In [ ]:
# Get catalogs for the 'hsc' collection without changing the class state
Catalogs.get_catalogs(collection="hsc")

## Inspecting Catalog Metadata

Before querying a catalog, it's important to understand what data it contains and how that data is organized. Catalog metadata includes information about the columns in the catalog (e.g., their names, data types, and descriptions) as well as the capabilities of the catalog (e.g., what types of queries it supports). This information can help you construct effective queries and interpret the results correctly.

### Inspecting Catalog Columns

Use the `get_column_metadata()` method to inspect the columns of a catalog. This returns an `astropy.table.Table` with information about each column, including its name, data type, unit, and a description. This metadata is crucial for constructing valid queries, selecting columns of interest, and understanding which columns support different criteria syntax.

Again, you can specify the collection and catalog explicitly as inputs to the function, or you can rely on the default values stored in the class attributes.

In [ ]:
# Get column metadata for the sciencemetadata catalog in the ullyses collection
Catalogs.get_column_metadata(collection='ullyses', catalog='sciencemetadata')

If you only specify a collection, the default catalog for that collection will be used. Below, we specify the collection "hsc" to get column metadata for "dbo.SumMagAper2CatView", which is the default catalog for the "hsc" collection.

In [ ]:
# Get column metadata for the default catalog in the hsc collection
Catalogs.get_column_metadata(collection='hsc')

### Inspecting Catalog Capabilities

Each catalog has different capabilities, which are important to understand when constructing your queries. For example, only certain catalogs support spatial queries based on a sky position or region. Use the `supports_spatial_queries()` method to check if a catalog supports spatial queries.

In [ ]:
# Check if the 'ullyses' collection and 'sciencemetadata' catalog support spatial queries
Catalogs.supports_spatial_queries(collection='ullyses', catalog='sciencemetadata')

The Ullyses science metadata catalog does not support spatial queries. This means that if we try to pass any spatial parameters to a query, an error will be raised.

In [ ]:
# Check if the 'hsc' collection and supports spatial queries
Catalogs.supports_spatial_queries(collection='hsc')

The HSC summary source catalog does support spatial queries! We can use spatial query parameters in our queries to this catalog.

## Querying Catalogs

The `Catalogs` class provides three main methods for querying catalogs:

- `query_criteria` is the most flexible method. It supports purely column-based queries, purely positional queries, or a combination of both.
- `query_region` is a convenience method for positional queries that use coordinates or a region on the sky.
- `query_object` is a convenience method for positional queries that use an object name resolved to coordinates.

All three methods ultimately construct and execute an ADQL query against the MAST TAP service. They return results from a MAST catalog as an `astropy.table.Table` object (or a count of results if `count_only=True` is specified). All three support column-based filtering, sorting, and limiting of results. The primary difference between them is whether and how positional criteria are specified.

### Shared Query Parameters

The following parameters are shared across all three query methods:

- `collection`: The name of the catalog collection to query. If not specified, the `collection` class attribute will be used as the default.
- `catalog`: The name of the catalog to query within the specified collection. If not specified, the `catalog` class attribute will be used as the default.
- `limit`: An integer specifying the maximum number of results to return. The default is 5,000.
- `offset`: An integer specifying the number of results to skip before starting to return results. This is useful for paginating through large result sets. Default is 0.
- `count_only`: A boolean indicating whether to return only the count of matching results instead of the results themselves. Default is False.
- `select_cols`: A list of column names to include in the results. If not specified, all columns will be returned.
- `sort_by`: A string or list of strings specifying the column(s) to sort the results by. Default is None (no sorting).
- `sort_desc`: A boolean or list of booleans specifying whether to sort in descending order for each column specified in `sort_by`. Default is False (ascending order).

### Criteria Syntax

All query methods also allow you to filter results based on column values. Users may specify criteria using keyword arguments, where the keyword is the column name and the value is the filter condition. Multiple criteria are combined using a logical AND. 

Criteria syntax supports a variety of operations for filtering results:

- A single value (e.g. `column=value`) will filter for rows where the column is equal to the value.
- A list of values (e.g. `column=[value1, value2]`) will filter for rows where the column is equal to any of the values in the list (logical OR).
- A value prefixed with `!` (e.g. `column="!value"`) will filter for rows where the column is not equal to the value.
- For string columns, a string with a wildcard character `*` (e.g. `column="NGC*"`) will filter for rows where the column value matches the pattern, where `*` can match any sequence of characters.
- For numeric columns, a string with a comparison operator (e.g. `column=">value"`) will filter for rows where the column value satisfies the specified comparison. Supported operators include `<`, `>`, `<=`, and `>=`.
- For numeric columns, a string with a range (e.g. `column="value1..value2"`) will filter for rows where the column value falls within the specified range (inclusive).
- For temporal columns, values can be specified as strings in a recognized date/time format (e.g. `YYYY`, `YYYY-MM-DD`, `YYYY-MM-DD hh:mm:ss`, etc.), `astropy.time.Time` objects, or `datetime` objects. The same comparison operators and range syntax as numeric columns can be used to filter temporal columns based on date/time values.

We'll use the Ullyses science metadata catalog to demonstrate a column-based query, since it doesn't support spatial queries. Let's filter for the following:

- Targets with a name that starts with "NGC".
- Targets that belong to spectral class "O" or "B".
- Targets that are NOT known binaries.
- Targets that are NOT classified as "Galaxy" or "Late O Dwarf".
- Targets with Gaia parallax less than -0.01 or greater than or equal to 0.
- Targets with effective temperature between 30,000 and 50,000 K.

We will also select a subset of columns to return with the ``select_cols`` parameter.

In [ ]:
# Ullyses query with multiple non-positional criteria
select_cols = ["target_name_ullyses", "target_classification", "known_binary", "sp_class", "gaia_parallax", "star_teff"]
Catalogs.query_criteria(
    collection="ullyses",  # Query the 'ullyses' collection
    catalog="sciencemetadata", # Query the 'sciencemetadata' catalog
    target_name_ullyses="NGC*",  # Query for targets names starting with 'NGC'
    sp_class=["O", "B"],  # Query for targets with spectral class 'O' or 'B'
    known_binary=False,  # Query for targets that are not known binaries
    target_classification=["!Galaxy", "!Late O Dwarf"],  # Exclude targets classified as 'Galaxy' or 'Late O Dwarf'
    gaia_parallax=["<-0.01", ">=0"],  # Query for targets with Gaia parallax less than -0.01 or greater than or equal to 0
    star_teff="30000..50000",  # Query for targets with effective temperature between 30,000 and 50,000 K
    select_cols=select_cols
)

### Spatial Query Parameters

If a catalog supports spatial queries, the following parameters can be used to specify the spatial region of interest:

- `coordinates`: A string or `astropy.coordinates.SkyCoord` object specifying the center of a cone search. This parameter is used in conjunction with `radius`.
- `object_name`: A string specifying the name of an astronomical object to resolve to coordinates for a cone search. This parameter is used in conjunction with `radius`.
- `resolver`: A string specifying the name of the resolver to use when resolving `object_name` to coordinates. This is only applicable when `object_name` is provided. Default is None.
- `radius`: The radius of a cone search around `coordinates` or `object_name`. Can be defined as a string with units (e.g., "10 arcsec"), a `astropy.units.Quantity`, or a float in degrees. Default is 0.2 degrees.
- `region`: Specifies the spatial region of interest for more complex spatial queries, such as polygon searches. Please see the [Specifying Spatial Regions](#specifying-spatial-regions) section below for details on how to use this parameter.

If no spatial parameters are provided, the query is purely column-based and will not filter results based on position. If they are supplied, the spatial parameters are combined with any column-based criteria using a logical AND, meaning that only results that satisfy both the spatial and column-based criteria will be returned.

We'll demonstrate a spatial query using the HSC summary source catalog, which supports spatial queries. Let's filter for the following:

- Sources within 2 arcseconds of the coordinates (322.49324, 12.16683).
- Sources with target names that are either "M-15" or start with "NGC".
- Sources with a start time between 2006 and 2013.

The query will sort results first by the number of images in ascending order, and then by start time in descending order. We'll also limit the number of results returned to 10 and select a subset of columns to return with the `select_cols` parameter.

In [ ]:
# HSC position-based query with multiple criteria and sorting
Catalogs.query_criteria(
    collection="hsc",  # Query the 'hsc' collection
    coordinates=SkyCoord("322.49324 12.16683", unit='deg'),
    radius="2 arcsec",  # Query for sources within 2 arcseconds of the specified coordinates
    sort_by=["numimages", "starttime"],  # Sort results by number of images and then by starttime
    sort_desc=[False, True],  # Sort numimages in ascending order and starttime in descending order
    targetname=["M-15", "NGC*"],  # Query for targets with names 'M-15' or starting with 'NGC'
    starttime="2006..2013",  # Query for observations with starttime between 2006 and 2013
    limit=10,  # Limit to 10 results
    select_cols=["matchid", "matchra", "matchdec", "numimages", "starttime", "targetname"]
)

The `query_region` and `query_object` methods are convenience methods for spatial queries. `query_region` allows you to specify a region on the sky using the `coordinates`, `radius`, and/or `region` parameters. `query_object` allows you to specify an `object_name` that will be resolved to coordinates and a `radius` for a cone search. Both methods also support column-based criteria, sorting, and limiting of results, just like `query_criteria`.

For these queries, we will use the "skymapperdr4" collection, which contains catalogs from the [SkyMapper Southern Survey: Data Release 4](https://skymapper.anu.edu.au/data-release/). The catalog we query will be "dr4.master", which is a master catalog of sources detected in the SkyMapper DR4 survey, containing their positions, magnitudes, and other properties. We'll use the `query_region` method to perform a simple cone search for sources within 1 arcminute of the coordinates (158.47924, -7.30962).

In [ ]:
# Skymapper DR4 region query with coordinates and radius
Catalogs.query_region(
    collection='skymapperdr4',
    catalog='dr4.master',
    coordinates=SkyCoord("158.47924 -7.30962", unit='deg'),
    radius='1 arcmin',
    select_cols=['object_id', 'raj2000', 'dej2000']
)

For this next query, we'll use the `query_object` method to search for sources within 0.1 degrees of the object ["M11"](https://science.nasa.gov/mission/hubble/science/explore-the-night-sky/hubble-messier-catalog/messier-11/), which is an open star cluster also known as the Wild Duck Cluster. We'll also filter for sources with a U-band PSF magnitude greater than 10, sort results by the mean epoch of the observation, and limit the number of results to 10.

In [ ]:
# Skymapper DR4 object query with additional criteria
Catalogs.query_object(
    collection='skymapperdr4',
    object_name='M11',
    radius=0.1,
    u_psf=">10",
    sort_by='mean_epoch',
    select_cols=['object_id', 'raj2000', 'dej2000', 'u_psf', 'mean_epoch'],
    limit=10
)

#### Specifying Spatial Regions

For catalogs that support spatial queries, there are several ways to specify the spatial region of interest. The simplest is a cone search, defined by a center position and a radius. More complex regions, such as polygons, can also be specified using the `region` parameter.

##### Cone Search

Cone searches are the most common type of spatial query, defined by a center position and a radius. They may be specified using:

- The `coordinates` and `radius` parameters together.
- The `object_name` and `radius` parameters together, where the object name is resolved to coordinates.
- The `region` parameter as:
  - A `regions.CircleSkyRegion` object from the `regions` package.
  - A Space-Time Coordinate (STC) string in the format `CIRCLE [frame] <ra> <dec> <radius>`, where ra, dec, and radius are in degrees.

Let's demonstrate this on the `caom` collection, which refers to the [Common Archive Observation Model](https://www.ivoa.net/documents/CAOM/20240927/WD-CAOM-2.5-20240927.pdf), a data model used to describe astronomical observations. We'll query the `obspointing` catalog, which contains metadata about scientific observations at MAST. Our cone search will be centered on the coordinate (18.855, -6.945) with a radius of 0.01 degrees.

In [ ]:
# Search for sources in the CAOM observations catalog that fall within a circular region
Catalogs.query_region(
    collection="caom",
    catalog="obspointing",
    region="CIRCLE ICRS 18.855 -6.945 0.01",
    limit=5,
    select_cols=["target_name", "obs_id", "s_ra", "s_dec", "s_region"]
)

#### Polygon Search

Polygon searches allow for more complex spatial queries by defining a polygonal region on the sky. They may be specified using any of the following as the ``region`` parameter:

- An iterable of (RA, Dec) tuples representing the vertices of the polygon.
- A `regions.PolygonSkyRegion` object from the `regions` package.
- A Space-Time Coordinate (STC) string in the format `POLYGON [frame] <ra_1> <dec_1> <ra_2> <dec_2> ... <ra_n> <dec_n>`, where (ra_i, dec_i) are the vertices of the polygon in degrees.

Keep in mind that at least three vertices are required to define a valid polygon, and the vertices should be ordered either clockwise or counterclockwise. The polygon will be automatically closed by connecting the last vertex back to the first.

Let's search for sources in the CAOM observations catalog that fall within a four-sided polygon. The polygon has the same center as the previous cone search, so the results may look similar!

In [ ]:
# Search for sources in the CAOM observations catalog that fall within a four-sided polygonal region
Catalogs.query_region(
    collection="caom",
    catalog="obspointing",
    region="POLYGON ICRS 18.85 -6.95 18.86 -6.95 18.86 -6.94 18.85 -6.94",
    limit=5,
    select_cols=["target_name", "obs_id", "s_ra", "s_dec", "s_region"],
)

### Counting Results

Each of the three query methods supports a ``count_only`` parameter. When set to ``True``, it returns only the number of matching results. This can be useful for quickly assessing the size of a result set without having to retrieve all the data.

Below, we demonstrate this on the `tic_v82` collection, which refers to the [TESS Input Catalog version 8.2](https://ui.adsabs.harvard.edu/abs/2022yCat.4039....0P/abstract). We'll perform a query to count the number of TESS sources that are within 0.1 degrees of the exoplanet [WASP-12b](https://science.nasa.gov/exoplanet-catalog/wasp-12-b/). You'll notice that this query takes several seconds to complete. If you were to run the same query without the ``count_only`` parameter, it would take significantly longer to return the full results, especially if there are many matching sources. Feel free to try it out and see the difference in execution time!

In [ ]:
# Counting results for a query with multiple criteria
Catalogs.query_criteria(
    collection="tic_v82",
    catalog="tic_v82.source",
    object_name="WASP-12b",
    radius=0.1,
    count_only=True
)

## Exercises

**Exercise 1**: It's time to apply all that you've learned and try your hand at writing a `Catalogs` query! For this exercise, we'll be working with the "gaiadr3" collection, which contains catalogs from [Gaia Data Release 3](https://www.cosmos.esa.int/web/gaia/dr3). The catalog we will query is "gaia_source", which is the main Gaia source catalog containing astrometric and photometric data for over 1.8 billion sources.

Start by creating an instance of the `Catalogs` class with the `collection` and `catalog` attributes set accordingly. Then, use that object to get the column metadata and check if the catalog supports spatial queries.

In [ ]:
# Create a Catalogs object for the Gaia source catalog
# gaia_catalog = ...

In [ ]:
# Get column metadata for the Gaia source catalog
# ...

In [ ]:
# Check if the Gaia source catalog supports spatial queries
# ...

**Exercise 2**: Using your new `Catalogs` object for the Gaia source catalog, write a query to find all sources that meet the following criteria:

- Sources with proper motion in the range of 1 to 10 mas/yr or greater than or equal to 15 mas/yr.
- Sources with a designation that contains "820" but does not contain "363".
- Sources that are part of the [Andromeda Photometric Survey](https://arxiv.org/abs/2206.05591).
- Sources with a class probability of being a star (`classprob_dsc_combmod_star`) greater than 0.99.

The query should return the `solution_id`, `designation`, `pm`, `in_andromeda_survey`, and `classprob_dsc_combmod_star` columns, sorted by `solution_id` in descending order. Limit the number of results returned to 100.

In [ ]:
# Query the Gaia source catalog with multiple criteria
# gaia_catalog.query_criteria(...)

**Exercise 3**: Now let's try a spatial query! Write a query to find all the sources in the Gaia source catalog that meet the following criteria:

- Sources within a polygonal sky region defined by the vertices (10, 10), (10, 11), (11, 12), and (11, 10).
- Sources with a class probability of being a galaxy (`classprob_dsc_combmod_galaxy`) greater than 0.99. 
- Sources with a proper motion that is NOT between 2.5 and 4 mas/yr.

Select the `solution_id`, `designation`, `pm`, and `classprob_dsc_combmod_galaxy` columns.

In [ ]:
# Query with spatial criteria and non-positional criteria
# gaia_catalog.query_criteria(...)

**Exercise 4**: Almost there! Your final exercise is to write a count-only query to find out how many sources in the Gaia source catalog have a class probability of being a quasar (`classprob_dsc_combmod_quasar`) greater than 0.99.

In [ ]:
# Count-only query for sources with a class probability of being a quasar greater than 0.99
# gaia_catalog.query_criteria(...)

## Exercise Solutions

**Exercise 1:**

In [ ]:
# Create a Catalogs object for the Gaia source catalog
gaia_catalog = Catalogs(collection="gaiadr3", catalog="gaia_source")

In [ ]:
# Get column metadata for the Gaia source catalog
gaia_catalog.get_column_metadata()

In [ ]:
# Check if the Gaia source catalog supports spatial queries
gaia_catalog.supports_spatial_queries()

**Exercise 2:**

In [ ]:
# Query the Gaia source catalog with multiple criteria
gaia_catalog.query_criteria(
    pm=["1..10", ">=15"],
    designation=["*820*", "!363*"],
    in_andromeda_survey=True,
    classprob_dsc_combmod_star=">0.99",
    limit=100,
    sort_by="solution_id",
    sort_desc=True,
    select_cols=["solution_id", "designation", "pm", "in_andromeda_survey", "classprob_dsc_combmod_star"]
)

**Exercise 3:**

In [ ]:
# Query with spatial criteria and non-positional criteria
gaia_catalog.query_criteria(
    region=[(10, 10), (10, 11), (11, 12), (11, 10)],
    classprob_dsc_combmod_galaxy=">0.99",
    pm="!2.5..4",
    select_cols=["solution_id", "designation", "pm", "classprob_dsc_combmod_galaxy"],
)

**Exercise 4:**

In [ ]:
# Count-only query for sources with a class probability of being a quasar greater than 0.99
gaia_catalog.query_criteria(
    classprob_dsc_combmod_quasar=">0.99",
    count_only=True
)

## Additional Resources

- [`astroquery.mast` Documentation for Catalog Searches](https://astroquery.readthedocs.io/en/latest/mast/mast_catalog.html)
- [MAST TAP Service Documentation](https://mast.stsci.edu/vo-tap/)
- [MAST Catalogs Search UI](https://mast.stsci.edu/catalogs/#/search)

## Citations

If you use `astroquery` for published research, please cite the
authors. Follow these links for more information about citing `astroquery`:

* [Citing Astroquery](https://github.com/astropy/astroquery/blob/main/astroquery/CITATION)

## About this Notebook

**Author(s):** Sam Bianco <br>
**Keyword(s):** Tutorial, Astroquery, Catalogs, VO-TAP <br>
**First published:** May 2026<br>
**Last updated:** May 2026 <br>

***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 